# CP493: finish the H4 runs on a Kaggle GPU

Runs the same code as the Mac, from the GitHub repo, on the BraTS 2021 data attached on Kaggle.

**Before running**
1. Settings (right panel) → **Accelerator: GPU T4 x2**, **Internet: On** (needs a phone-verified Kaggle account).
2. **Add Input** → search `brats-2021-task1` (by dschettler8845) → Add.
3. Only if the GitHub repo is private: **Add-ons → Secrets** → add `GITHUB_TOKEN` (a fine-grained GitHub token with read access to this repo) and tick it for this notebook.
4. **Save Version → Save & Run All (Commit)**. It keeps running after you close the browser.
5. When it finishes: **Output** tab → download `results_kaggle.zip`.

What it does: a 30-patient check that Kaggle gives the same numbers as the Mac, then either (same numbers) finishes the two
memory-adapter runs, or (different numbers) re-runs every H4 comparison here under `k_` names, so no comparison mixes machines.

In [ ]:
REPO = "https://github.com/Dharmik1411/cp493-sam2-brats.git"
MAX_PATIENT_DIFF = 0.01   # check passes if no patient's Dice differs from the Mac by more than this...
MAX_MEAN_DIFF = 0.002     # ...and the mean difference is below this

## 1. Code

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
url = REPO
try:  # private repo: use the GITHUB_TOKEN secret; public repo: no secret needed
    from kaggle_secrets import UserSecretsClient
    url = REPO.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
except Exception:
    pass
if not os.path.isdir("cp493-sam2-brats"):
    r = subprocess.run(["git", "clone", "--depth", "1", url, "cp493-sam2-brats"], capture_output=True, text=True)
    if r.returncode:  # don't print the command: it can hold the token
        raise SystemExit("git clone failed. Private repo? Add the GITHUB_TOKEN secret (see the top cell).")
os.chdir("/kaggle/working/cp493-sam2-brats")
need = ["lora_memory.pt", "lora_memory_long_lowlr.pt", "lora_encoder.pt", "lora_encoder_v2.pt", "lora_memory_v2.pt", "results/rp30_150.csv"]
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f"not on GitHub yet, push them from the Mac first: {missing}"
print("code ready")

## 2. Install (SAM 2 without its CUDA add-on, so hole filling is skipped exactly as on the Mac)

In [ ]:
!SAM2_BUILD_CUDA=0 pip install -q -r requirements.txt
import torch
print(torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "turn on the GPU T4 x2 accelerator"

## 3. Data: unpack the 150 test patients

In [ ]:
import glob, json, tarfile
want = set(json.load(open("splits_base.json"))["test"])
DATA = "/tmp/data"
os.makedirs(DATA, exist_ok=True)
unpacked = glob.glob("/kaggle/input/**/BraTS2021_00000", recursive=True)
if unpacked:  # the dataset is already unpacked on Kaggle: use it in place
    DATA = os.path.dirname(unpacked[0])
elif any(not os.path.isdir(os.path.join(DATA, c)) for c in want):
    tars = glob.glob("/kaggle/input/**/BraTS2021_Training_Data.tar", recursive=True)
    assert tars, "attach the dataset brats-2021-task1 (Add Input)"
    with tarfile.open(tars[0], "r|") as tar:  # one pass through the archive, keeping only the test patients
        if hasattr(tarfile, "data_filter"):
            tar.extraction_filter = tarfile.data_filter
        for m in tar:
            if m.name.lstrip("./").split("/")[0] in want:
                tar.extract(m, DATA)
ready = sum(len(glob.glob(os.path.join(DATA, c, "*.nii*"))) == 5 for c in want)
print(ready, "of", len(want), "test patients ready in", DATA)
assert ready == len(want)

## 4. Check: does Kaggle give the same numbers as the Mac? (zero-shot, first 30 patients, about 15 to 30 min)

In [ ]:
COMMON = f"--data {DATA} --splits splits_base.json --subset test --norm slice --regions WT --mode reprompt --placement largest --direction both --step 30 --resume"
!python run.py eval {COMMON} --limit 30 --name kcheck_rp30_30
import csv, numpy as np
k = {r["case"]: float(r["dice"]) for r in csv.DictReader(open("results/kcheck_rp30_30.csv"))}
m = {r["case"]: float(r["dice"]) for r in csv.DictReader(open("results/rp30_150.csv"))}
d = np.array([k[c] - m[c] for c in k])
same = abs(d.mean()) < MAX_MEAN_DIFF and np.abs(d).max() < MAX_PATIENT_DIFF
print(f"Kaggle {np.mean(list(k.values())):.4f} vs Mac {np.mean([m[c] for c in k]):.4f} mean Dice; "
      f"per-patient difference mean {d.mean():+.4f}, largest {np.abs(d).max():.4f} -> "
      + ("SAME: finish the two memory runs" if same else "DIFFERENT: re-run every H4 comparison here under k_ names"))

## 5. The H4 runs, split over the two GPUs (progress in log_gpu0.txt and log_gpu1.txt)

In [ ]:
import time
P = "" if same else "k_"
if same:  # memory v1 resumes from the patients the Mac already did
    jobs = [("lora_memory_rp30_150", "--lora lora_memory.pt"),
            ("lora_memory_long_lowlr_rp30_150", "--lora lora_memory_long_lowlr.pt")]
else:
    jobs = [("k_rp30_150", ""), ("k_lora_encoder_v2_rp30_150", "--lora lora_encoder_v2.pt"),
            ("k_lora_memory_rp30_150", "--lora lora_memory.pt"), ("k_lora_memory_v2_rp30_150", "--lora lora_memory_v2.pt"),
            ("k_lora_encoder_rp30_150", "--lora lora_encoder.pt"), ("k_lora_memory_long_lowlr_rp30_150", "--lora lora_memory_long_lowlr.pt")]
n_gpu = torch.cuda.device_count()
procs = []
for g in range(n_gpu):
    mine = jobs[g::n_gpu]
    cmd = " && ".join(f"python run.py eval {COMMON} --limit 150 {a} --name {n}" for n, a in mine)
    procs.append(subprocess.Popen(cmd, shell=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g)},
                                  stdout=open(f"/kaggle/working/log_gpu{g}.txt", "w"), stderr=subprocess.STDOUT))
start = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(600)
    rows = {n: max(sum(1 for _ in open(f"results/{n}.csv")) - 1, 0) if os.path.exists(f"results/{n}.csv") else 0 for n, _ in jobs}
    print(f"{(time.time() - start) / 3600:.1f} h:", rows, flush=True)
assert all(p.returncode == 0 for p in procs), "a run failed: see log_gpu0.txt / log_gpu1.txt"
print("all runs done")

## 6. H4 statistics (paired Wilcoxon, Holm, bootstrap CI)

In [ ]:
names = ["rp30_150", "lora_encoder_rp30_150", "lora_memory_rp30_150", "lora_encoder_v2_rp30_150", "lora_memory_v2_rp30_150", "lora_memory_long_lowlr_rp30_150"]
files = " ".join(f"results/{P}{n}.csv" for n in names)
!python analyze.py --all {files}

## 7. Save the results for download (Output tab → results_kaggle.zip)

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results_kaggle", "zip", "results")
print("saved /kaggle/working/results_kaggle.zip")